<a href="https://colab.research.google.com/github/surajsingh2182-bot/retail-dynamic-pricing-engine/blob/main/2_Retail_Dynamic_Pricing_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# Set seed for reproducible portfolio metrics
np.random.seed(42)
n_products = 2000

# 1. SCOPING THE FEATURE MATRIX (X) - Only variables available at page-load time
data = {
    'product_id': [f"PROD_{i:04d}" for i in range(n_products)],
    'competitor_price': np.random.uniform(15.00, 150.00, size=n_products).round(2),
    'inventory_stock_level': np.random.randint(5, 500, size=n_products),
    'demand_surge_multiplier': np.random.choice([1.0, 1.15, 1.30, 1.50], size=n_products, p=[0.70, 0.15, 0.10, 0.05]),
    'historical_views_past_24h': np.random.randint(10, 2500, size=n_products)
}

df = pd.DataFrame(data)

# 2. GENERATING THE TARGET VARIABLE (y) - The baseline market-clearing price
# Price is dynamically calculated based on competitive pressure, scarcity, and demand
df['optimal_target_price'] = (
    (df['competitor_price'] * 0.95) +
    (df['demand_surge_multiplier'] * 8.50) -
    (np.log1p(df['inventory_stock_level']) * 1.20) +
    np.random.normal(0, 3.5, size=n_products) # Adding natural market noise
).round(2)

# Ensure no negative prices due to random noise
df['optimal_target_price'] = df['optimal_target_price'].clip(lower=5.00)

print(f"✅ Success! Generated clean, leakage-free retail dataset: {df.shape[0]} products.")
print("\nPreview of the Retail Warehouse Feature Ledger:")
print(df[['product_id', 'competitor_price', 'inventory_stock_level', 'demand_surge_multiplier', 'optimal_target_price']].head())

✅ Success! Generated clean, leakage-free retail dataset: 2000 products.

Preview of the Retail Warehouse Feature Ledger:
  product_id  competitor_price  inventory_stock_level  \
0  PROD_0000             65.56                    224   
1  PROD_0001            143.35                    395   
2  PROD_0002            113.82                    167   
3  PROD_0003             95.82                    443   
4  PROD_0004             36.06                    224   

   demand_surge_multiplier  optimal_target_price  
0                      1.0                 68.82  
1                      1.3                138.42  
2                      1.3                116.11  
3                      1.0                 87.56  
4                      1.0                 37.18  


In [3]:
df['optimal_target_price'].describe()

,optimal_target_price
count,2000.000000
mean,81.082825
std,37.606971
min,9.840000
25%,47.815000
50%,82.800000
75%,113.200000
max,151.240000


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. ISOLATE FEATURES AND TARGET
X = df[['competitor_price', 'inventory_stock_level', 'demand_surge_multiplier', 'historical_views_past_24h']]
y = df['optimal_target_price']

# 2. THE PM STRATEGY: Handle Outliers via Log Transformation on Target (y)
y_log = np.log1p(y)

# 3. SPLIT DATA INTO TRAIN AND TEST SETS (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

# 4. PREPROCESSING: Standardize Input Features (X)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. MODEL TRAINING: Initialize and fit our Linear Regression baseline
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# 6. INFERENCE & REVERSING THE LOG CHANGE FOR VALIDATION
predictions_log = model.predict(X_test_scaled)
# Convert log numbers back into real currency dollars using exponential function (expm1)
predictions_actual = np.expm1(predictions_log)
y_test_actual = np.expm1(y_test)

# 7. REGRESSION METRIC EVALUATION
rmse = np.sqrt(mean_squared_error(y_test_actual, predictions_actual))
r2 = r2_score(y_test_actual, predictions_actual)

print("📊 Baseline Dynamic Pricing Model Metrics:")
print(f"• Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"• R-Squared (R²) Score: {r2:.4f} ({r2*100:.1f}% of price variance explained)")

📊 Baseline Dynamic Pricing Model Metrics:
• Root Mean Squared Error (RMSE): $12.08
• R-Squared (R²) Score: 0.8984 (89.8% of price variance explained)


In [5]:
def production_pricing_gateway(base_competitor_price, inventory, surge_multiplier, historical_views, manufacturing_cost):
    # 1. Format the real-time storefront variables into an array matching our training matrix
    raw_features = pd.DataFrame([{
        'competitor_price': base_competitor_price,
        'inventory_stock_level': inventory,
        'demand_surge_multiplier': surge_multiplier,
        'historical_views_past_24h': historical_views
    }])

    # 2. Apply our standard scaling transformations matching our training pipeline
    scaled_features = scaler.transform(raw_features)

    # 3. Generate the unconstrained mathematical price from the model
    predicted_log_price = model.predict(scaled_features)[0]
    unconstrained_ml_price = float(np.expm1(predicted_log_price))

    # 4. ENFORCE PM OPERATIONAL GUARDRAILS (Choice B)
    # Ensure storefront never prices below manufacturing cost + a minimum 15% safety margin
    minimum_floor_price = manufacturing_cost * 1.15
    final_storefront_price = max(unconstrained_ml_price, minimum_floor_price)

    # 5. Output structured transaction log for system monitoring
    return {
        "raw_ml_prediction": round(unconstrained_ml_price, 2),
        "regulatory_cost_floor": round(minimum_floor_price, 2),
        "final_storefront_listed_price": round(final_storefront_price, 2),
        "guardrail_triggered": final_storefront_price == minimum_floor_price
    }

# --- SIMULATE STOREFRONT SCENARIO ---
# Scenario: Competitor drops price to $10.00, risking pulling our ML model beneath our $15.00 production cost
print("🛒 Storefront Real-Time Inference Evaluation:")
api_response = production_pricing_gateway(
    base_competitor_price=10.00,
    inventory=450,
    surge_multiplier=1.0,
    historical_views=15,
    manufacturing_cost=15.00
)
import json
print(json.dumps(api_response, indent=4))

🛒 Storefront Real-Time Inference Evaluation:
{
    "raw_ml_prediction": 24.96,
    "regulatory_cost_floor": 17.25,
    "final_storefront_listed_price": 24.96,
    "guardrail_triggered": false
}


In [6]:
# 1. ENGINEER THE DERIVED SCARCITY FEATURE SIGNAL (Choice C)
# Adding a small constant (+1) to prevent a fatal division-by-zero error if views are 0
df['inventory_to_demand_ratio'] = df['inventory_stock_level'] / (df['historical_views_past_24h'] + 1)

# 2. UPDATE THE FEATURE MATRIX MATRIX
X_advanced = df[['competitor_price', 'inventory_stock_level', 'demand_surge_multiplier', 'historical_views_past_24h', 'inventory_to_demand_ratio']]
y_advanced_log = np.log1p(df['optimal_target_price'])

# 3. SPLIT TRAIN/TEST (80/20)
X_train_adv, X_test_adv, y_train_adv, y_test_adv = train_test_split(X_advanced, y_advanced_log, test_size=0.2, random_state=42)

# 4. SCALE THE ENTIRE ADVANCED ARRAY
scaler_adv = StandardScaler()
X_train_adv_scaled = scaler_adv.fit_transform(X_train_adv)
X_test_adv_scaled = scaler_adv.transform(X_test_adv)

# 5. RETRAIN THE PRICING ENGINE
advanced_model = LinearRegression()
advanced_model.fit(X_train_adv_scaled, y_train_adv)

# 6. RUN INFERENCE AND INVERT LOG ARTIFACTS
adv_preds_log = advanced_model.predict(X_test_adv_scaled)
adv_preds_actual = np.expm1(adv_preds_log)
y_test_adv_actual = np.expm1(y_test_adv)

# 7. LOG ADVANCED portfolio COMPLIANCE METRICS
adv_rmse = np.sqrt(mean_squared_error(y_test_adv_actual, adv_preds_actual))
adv_r2 = r2_score(y_test_adv_actual, adv_preds_actual)

print("📈 Advanced Dynamic Pricing Model Metrics (With Scarcity Ratio):")
print(f"• Optimised Root Mean Squared Error (RMSE): ${adv_rmse:.2f}")
print(f"• Optimised R-Squared (R²) Score: {adv_r2:.4f} ({adv_r2*100:.1f}% explained)")

# Calculate the financial improvement edge
print(f"\n🎯 Strategic Product Impact: Introducing the scarcity ratio adjusted error variance down by ${abs(rmse - adv_rmse):.2f} per product entry!")

📈 Advanced Dynamic Pricing Model Metrics (With Scarcity Ratio):
• Optimised Root Mean Squared Error (RMSE): $12.08
• Optimised R-Squared (R²) Score: 0.8985 (89.8% explained)

🎯 Strategic Product Impact: Introducing the scarcity ratio adjusted error variance down by $0.00 per product entry!
